# v27 実験環境 — 同一typewellのHW間GR照合（フェーズ1: train/valid 分割）

## 仮説
一部の坑井は **typewell が実質同一（ファイル名だけ違う）** で、horizontal well(HW) だけが異なる。
その場合、**train HW は未知区間(eval)も真値 TVT が既知**という特性を使い、
**train HW の GR と test HW の GR を直接マッチング**して、test HW の eval 区間 TVT を高精度に当てられるのでは、という仮説。
（＝現行 PF/GBM とは別系統。同一typewellを持たない坑井は対象外＝現行パイプラインで処理。）

## このノートの範囲（フェーズ1）
1. 全 train typewell を**全列・全行の完全一致**でグルーピング（ハッシュで束ね、`pandas.equals` で検証）。
2. **グループサイズ≥2**（＝同一typewellの相方がいる）坑井だけを対象に、
   **グループ内 leave-one-well-out (OOF)** で `valid坑井 ↔ 参照train坑井` のペアを作る。
   - valid坑井: test を模擬（GR全区間 + TVT_input既知prefix のみ与え、eval区間TVTは採点用の隠し真値）。
   - 参照坑井: train全既知を模擬（GR + **TVT全真値**）。
3. データアクセサ＋**採点土台**（eval行プールの overall RMSE = Kaggle指標）を用意。
   **照合ロジックは今後実装** → `match_predict()` をスタブで置き、ベースライン(last_known保持)で土台の動作確認だけする。

## 注意
- overall RMSE は全 valid 坑井の eval 行を1つにプールして算出（[[CLAUDE.md §4]]）。
- 対象は**同一typewellグループのみ**。全体に対するカバレッジ（坑井数/行数）も報告する。
- 同一判定は**完全一致（丸めなし・全列）**。`typewell` は名前が違うだけで中身が全く同じ、という前提。


In [ ]:
# ===== import + データパス自動判定 + train坑井一覧 =====
import os, glob, json, time, hashlib, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TRAIN = COMP / "train"
ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v27\artifacts_v27"); ART.mkdir(parents=True, exist_ok=True)

WELLS = sorted(os.path.basename(p).split("__")[0]
               for p in glob.glob(str(TRAIN / "*__horizontal_well.csv")))
print("COMP:", COMP, "| train wells:", len(WELLS), "| ART:", ART)

In [ ]:
# ===== typewell フィンガープリント(完全一致判定: 全列・全行) =====
# 同一typewellは「ファイル名が違うだけで中身は全く同じ」→ 丸めず全内容をハッシュ化して等価判定。
# (ハッシュはグルーピング用の高速化。実際の完全一致は次セルで pandas.equals で検証する)
def fingerprint_df(df):
    return hashlib.sha1(df.to_csv(index=False).encode("utf-8")).hexdigest()[:16]

TW_CACHE = {}; fp_of = {}; info = {}; t0 = time.time()
for k, wid in enumerate(WELLS):
    try:
        tw = pd.read_csv(TRAIN / f"{wid}__typewell.csv")
        TW_CACHE[wid] = tw
        fp_of[wid] = fingerprint_df(tw)
        gr = tw["GR"].to_numpy(float); tvt = tw["TVT"].to_numpy(float)
        info[wid] = dict(n=len(tw), tvt_min=float(np.nanmin(tvt)), tvt_max=float(np.nanmax(tvt)),
                         gr_mean=float(np.nanmean(gr)), gr_std=float(np.nanstd(gr)))
    except Exception as e:
        print("fp fail", wid, e)
    if (k + 1) % 200 == 0: print(f"  {k+1}/{len(WELLS)} ({time.time()-t0:.0f}s)")
print(f"fingerprinted {len(fp_of)} typewells ({time.time()-t0:.0f}s) | 完全一致(全列全行)ハッシュ")

In [ ]:
# ===== 同一typewell でグルーピング + pandas.equals 検証 + 統計 =====
from collections import defaultdict, Counter
groups = defaultdict(list)
for wid, fp in fp_of.items():
    groups[fp].append(wid)
groups = {fp: sorted(ws) for fp, ws in groups.items()}

# --- 検証: 同一ハッシュのグループが本当に完全一致か pandas.equals で確認(衝突/誤検出チェック) ---
mismatch = []
for fp, ws in groups.items():
    if len(ws) < 2: continue
    base = TW_CACHE[ws[0]]
    for w in ws[1:]:
        if not base.equals(TW_CACHE[w]): mismatch.append((fp, ws[0], w))
print("pandas.equals 検証:",
      "全グループ完全一致 OK" if not mismatch else f"★不一致 {len(mismatch)}件! {mismatch[:3]}")

sizes = Counter(len(ws) for ws in groups.values())
multi = {fp: ws for fp, ws in groups.items() if len(ws) >= 2}   # 相方あり=対象
excl  = [ws[0] for ws in groups.values() if len(ws) == 1]        # 単独=対象外
n_multi_wells = sum(len(ws) for ws in multi.values())

print("=== typewell グループ統計 ===")
print(f"  総坑井           : {len(fp_of)}")
print(f"  ユニーク typewell : {len(groups)}")
print(f"  グループサイズ分布 : {dict(sorted(sizes.items()))}")
print(f"  同一typewellグループ(≥2): {len(multi)}  → 対象坑井 {n_multi_wells} 本")
print(f"  単独(対象外)      : {len(excl)} 本")
print(f"  ★対象カバレッジ   : {n_multi_wells}/{len(fp_of)} = {100*n_multi_wells/max(len(fp_of),1):.1f}% の坑井")

print("\n=== グループ例(サイズ上位5) ===")
for fp, ws in sorted(multi.items(), key=lambda kv: -len(kv[1]))[:5]:
    i0 = info.get(ws[0], {})
    print(f"  [{fp}] {len(ws)}本: {ws[:6]}{'...' if len(ws)>6 else ''}  "
          f"(n={i0.get('n')}, TVT[{i0.get('tvt_min'):.0f},{i0.get('tvt_max'):.0f}])")

# 保存
(ART / "typewell_groups.json").write_text(json.dumps(
    {"match": "exact_full", "verified_equal": (len(mismatch) == 0), "groups": groups,
     "multi": {fp: ws for fp, ws in multi.items()}, "excluded_singletons": excl}, ensure_ascii=False))
print("\nsaved:", ART / "typewell_groups.json")

In [ ]:
# ===== グループ内 leave-one-well-out (OOF) で valid↔参照ペアを作成 =====
oof_pairs = []   # (valid_wid, [ref_wids], group_fp, group_size)
for fp, ws in multi.items():
    for i, vw in enumerate(ws):
        refs = [w for j, w in enumerate(ws) if j != i]
        oof_pairs.append(dict(valid=vw, refs=refs, group=fp, group_size=len(ws)))

pairs_df = pd.DataFrame([{"valid": p["valid"], "n_refs": len(p["refs"]),
                          "group": p["group"], "group_size": p["group_size"],
                          "refs": "|".join(p["refs"])} for p in oof_pairs])
print("OOF ペア数(=対象valid坑井延べ):", len(oof_pairs))
print("valid坑井のユニーク数:", pairs_df["valid"].nunique())
print(pairs_df.head(10).to_string(index=False))

pairs_df.to_csv(ART / "oof_pairs.csv", index=False)
(ART / "oof_pairs.json").write_text(json.dumps(oof_pairs, ensure_ascii=False))
print("saved:", ART / "oof_pairs.csv", "/", ART / "oof_pairs.json")

In [ ]:
# ===== データアクセサ + 整合性チェック =====
# valid坑井: test模擬(GR全区間 + TVT_input既知 + eval区間の隠し真値TVT)
# 参照坑井 : train全既知模擬(GR + TVT全真値)
def _load_hw(wid):
    return pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv")

def ref_arrays(wid):
    hw = _load_hw(wid)
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float), tvt=hw["TVT"].to_numpy(float))   # 全真値

def valid_arrays(wid):
    hw = _load_hw(wid)
    ev = hw["TVT_input"].isna().to_numpy()
    last_kn = hw.loc[hw["TVT_input"].notna(), "TVT_input"]
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float),
                tvt_input=hw["TVT_input"].to_numpy(float),   # 既知prefix(eval=NaN)
                tvt_true=hw["TVT"].to_numpy(float),          # 採点用の隠し真値
                eval_mask=ev,
                last_known_tvt=float(last_kn.iloc[-1]) if len(last_kn) else np.nan,
                ids=[f"{wid}_{i}" for i in range(len(hw))])

# --- サンプル1ペアで整合性確認 ---
_p = oof_pairs[0]; _va = valid_arrays(_p["valid"]); _rf = ref_arrays(_p["refs"][0])
assert fp_of[_p["valid"]] == fp_of[_p["refs"][0]], "同一typewellでない!"
print("sample pair:", _p["valid"], "<- ref", _p["refs"][0], "(group_size", _p["group_size"], ")")
print("  valid: rows", len(_va["md"]), "| eval行", int(_va["eval_mask"].sum()),
      "| 既知prefix", int((~_va["eval_mask"]).sum()), "| last_known", round(_va["last_known_tvt"],2))
print("  ref  : rows", len(_rf["md"]), "| TVT真値 finite", int(np.isfinite(_rf["tvt"]).sum()))
print("  typewell fp 一致:", fp_of[_p["valid"]] == fp_of[_p["refs"][0]])

In [ ]:
# ===== 採点土台: matcher を差し替えて overall RMSE(eval行プール)を測る =====
def rmse_pool(err):
    e = err[np.isfinite(err)]; return float(np.sqrt(np.mean(e*e))) if len(e) else float("nan")

# ---- ★ここを今後実装（train HWのGR↔test HWのGR照合）----
def match_predict(va, refs):
    """valid坑井 va の eval 行の絶対TVTを、参照train坑井群 refs のGR-TVTを使って予測。
    戻り値: va['eval_mask'] が True の行に対応する絶対TVT配列(長さ=eval行数)。
    現状は未実装=None を返す(=採点はベースラインにフォールバック)。"""
    return None

def baseline_hold(va, refs):
    """ベースライン: 最終既知TVTを eval 区間そのまま保持(土台の動作確認用の床)。"""
    n = int(va["eval_mask"].sum())
    return np.full(n, va["last_known_tvt"], dtype=float)

def evaluate(matcher, pairs, max_wells=None, verbose=False):
    errs = []; per = []
    for k, p in enumerate(pairs):
        if max_wells is not None and k >= max_wells: break
        va = valid_arrays(p["valid"]); refs = [ref_arrays(r) for r in p["refs"]]
        pred = matcher(va, refs)
        if pred is None:
            pred = baseline_hold(va, refs)     # 未実装時フォールバック
        m = va["eval_mask"]; true = va["tvt_true"][m]
        pred = np.asarray(pred, float)
        if len(pred) != int(m.sum()):          # 長さ不一致は保持で埋める
            pred = baseline_hold(va, refs)
        e = pred - true; errs.append(e); per.append((p["valid"], rmse_pool(e)))
        if verbose: print(f"  {p['valid']}: eval {int(m.sum())} rmse {rmse_pool(e):.3f}")
    all_e = np.concatenate(errs) if errs else np.array([])
    return dict(overall=rmse_pool(all_e), n_wells=len(per), n_rows=len(all_e), per_well=per)

# --- ベースライン(last_known保持)で土台を確認。matcher実装後は evaluate(match_predict, oof_pairs) ---
res = evaluate(baseline_hold, oof_pairs)
print("=== ベースライン(last_known保持) on 同一typewell OOF ===")
print(f"  overall RMSE : {res['overall']:.4f}")
print(f"  valid坑井    : {res['n_wells']} 本 / eval行 {res['n_rows']}")
worst = sorted(res["per_well"], key=lambda kv: -(kv[1] if np.isfinite(kv[1]) else -1))[:5]
print("  悪い坑井上位5:", [(w, round(r,2)) for w, r in worst])
print("\n※ match_predict() を実装したら evaluate(match_predict, oof_pairs) で比較する。")

In [ ]:
# ===== 参照プール + カバレッジ・ゲート + M1: 構造 datum 転写 =====
# 発想: 同一typewell=同一ブロック。参照train坑井は U=TVT+Z(構造datum)の傾斜を真値で持つ。
# 参照の U(MD) を valid の MD へ補間し、valid の既知prefixで定数オフセットを較正→eval を予測。
# = PFのdip外挿を「隣接坑井の既知傾斜」で置換。参照MD外/参照TVT範囲外(トゥ)は NaN(=フォールバック)。
def _ref_pool_tvt(refs):
    rt = np.concatenate([r["tvt"] for r in refs]); rt = rt[np.isfinite(rt)]
    return float(np.min(rt)), float(np.max(rt))

def match_m1(va, refs, deg=0, min_prefix=20, tvt_margin=2.0):
    # ★実験結果(2026-07): deg0 は 10本群でまぐれ勝ちする坑井もあるが、10群プールでは hold 22.3 < M1 32.6 で悪化。
    #   同一typewell=同一「層序(GR-TVT)」であって同一「dip/幾何」ではない → MD整列は水平区間で dip差を積分し破綻。
    #   結論: この幾何ベース転写は非推奨。GR(層序)空間の照合(M2/PF augmented likelihood)へ。ここは比較用に残す。
    md = va["md"]; z = va["z"]; ev = va["eval_mask"]
    U_known = va["tvt_input"] + z                      # 既知U(eval=NaN)
    preds = []
    for r in refs:
        rmd = r["md"]; rU = r["tvt"] + r["z"]
        o = np.argsort(rmd); rmd = rmd[o]; rU = rU[o]
        keep = np.concatenate([[True], np.diff(rmd) > 0])   # 単調化(補間用)
        rmd = rmd[keep]; rU = rU[keep]
        Ui = np.interp(md, rmd, rU, left=np.nan, right=np.nan)  # 参照MD外=NaN
        kn = (~ev) & np.isfinite(U_known) & np.isfinite(Ui)
        if int(kn.sum()) < min_prefix: continue
        if deg == 0:
            corr = np.full(len(md), float(np.median(U_known[kn] - Ui[kn])))  # 定数較正
        else:
            corr = np.polyval(np.polyfit(md[kn], U_known[kn] - Ui[kn], deg), md)  # 1次=dip差も較正
        preds.append(Ui + corr)
    n_ev = int(ev.sum())
    if not preds: return np.full(n_ev, np.nan)
    Upred = np.nanmedian(np.vstack(preds), axis=0)      # 複数参照はU空間で中央値(平均は多峰で危険)
    pe = (Upred - z)[ev].astype(float)
    rtmin, rtmax = _ref_pool_tvt(refs)
    pe[(pe < rtmin - tvt_margin) | (pe > rtmax + tvt_margin)] = np.nan   # トゥ(範囲外)→NaN
    return pe

def gated(matcher, **kw):
    """matcher の NaN(トゥ/未カバー)を hold で埋めた完成predictorを返す(本番はPFにフォールバック)。"""
    def f(va, refs):
        pe = np.asarray(matcher(va, refs, **kw), float).copy()
        bad = ~np.isfinite(pe)
        if bad.any(): pe[bad] = va["last_known_tvt"]
        return pe
    return f
print("M1(構造datum転写) ready")

In [ ]:
# ===== M2: prior で枝を固定 → 参照(GR,TVT)から GR 最近傍で補正 =====
# 単一GR最近傍はPFと同じ多峰性で壊れる → prior の ±tvt_win に限定して「正しい枝」だけで照合。
# ★prior_fn は差し替え可能。M1(datum)は弱いので、将来は現行パイプラインのPF出力を prior に渡すのが本命。
def match_m2(va, refs, prior_fn=match_m1, tvt_win=15.0, min_win=5):
    prior = np.asarray(prior_fn(va, refs), float)          # NaN=トゥ/未カバー
    n_ev = int(va["eval_mask"].sum())
    rt = np.concatenate([r["tvt"] for r in refs]); rg = np.concatenate([r["gr"] for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt = rt[m]; rg = rg[m]
    o = np.argsort(rt); rt = rt[o]; rg = rg[o]             # TVT昇順(窓検索用)
    grv = va["gr"][va["eval_mask"]]
    out = prior.astype(float).copy()
    for i in range(n_ev):
        p = prior[i]
        if not np.isfinite(p): continue                    # トゥは触らない
        lo = int(np.searchsorted(rt, p - tvt_win)); hi = int(np.searchsorted(rt, p + tvt_win))
        if hi - lo < min_win: continue
        j = lo + int(np.argmin(np.abs(rg[lo:hi] - grv[i])))  # 枝内でGR最近傍
        out[i] = rt[j]
    return out
print("M2(GR照合, prior枝固定) ready")

In [ ]:
# ===== オラクル上限 + 手法比較(overall / covered / toe / cov%) =====
def eval_split(predictor, pairs):
    E = []; C = []
    for p in pairs:
        va = valid_arrays(p["valid"]); refs = [ref_arrays(r) for r in p["refs"]]
        ev = va["eval_mask"]; true = va["tvt_true"][ev]
        pred = np.asarray(predictor(va, refs), float)
        if len(pred) != len(true): pred = np.full(len(true), va["last_known_tvt"])
        rtmin, rtmax = _ref_pool_tvt(refs)
        cov = (true >= rtmin) & (true <= rtmax)            # covered=真値が参照TVT範囲内
        E.append(pred - true); C.append(cov)
    e = np.concatenate(E); c = np.concatenate(C)
    r = lambda x: float(np.sqrt(np.mean(x * x))) if len(x) else float("nan")
    return dict(overall=r(e), covered=r(e[c]), toe=r(e[~c]), cov=float(c.mean()), n=len(e))

def oracle_pred(va, refs):
    ev = va["eval_mask"]; true = va["tvt_true"][ev]
    rtmin, rtmax = _ref_pool_tvt(refs); cov = (true >= rtmin) & (true <= rtmax)
    return np.where(cov, true, va["last_known_tvt"])       # covered=完全, toe=hold(上限)

METHODS = [("hold(床)", baseline_hold),
           ("M1 datum転写(gated)", gated(match_m1)),
           ("M2 GR照合(gated)", gated(match_m2)),
           ("ORACLE上限", oracle_pred)]
SUBSETS = [("全34坑井", oof_pairs),
           ("2本ペア", [p for p in oof_pairs if p["group_size"] == 2]),
           ("10本群",  [p for p in oof_pairs if p["group_size"] >= 3])]
for sub_name, sub in SUBSETS:
    print(f"=== 同一typewell OOF [{sub_name}] valid {len(sub)}本 ===")
    print(f"  {'method':26s} {'overall':>8s} {'covered':>8s} {'toe':>8s} {'cov%':>6s}")
    for name, pred in METHODS:
        s = eval_split(pred, sub)
        print(f"  {name:26s} {s['overall']:8.3f} {s['covered']:8.3f} {s['toe']:8.3f} {s['cov']*100:5.1f}")
    print()
print("※ overall=eval行プール(Kaggle指標)。covered=参照TVT範囲内, toe=範囲外(=常に最深)。")
print("※ ★実験結論: M1(幾何datum転写)は hold より悪化(同一typewell=同一層序であって同一dipではない)。")
print("※ 有望性の判断は ORACLE(上限)と、現行パイプラインのPFをこの34坑井で測った値との比較で。")
print("※ 本命の次手: prior_fn に現行PF出力を渡した M2、または 参照(GR,TVT)で typewell尤度を augment した PF。")

In [ ]:
# ===== PF エンジン(v25 r0 torch/GPU 忠実移植) + 尤度grid(typewell / +疑似typewell) =====
# v25 submit_smoother の r0 torch エンジンをそのまま持込み(numpy移植でなくGPUバッチ=v25と等価)。
# grid(gg) を typewell / 混合(疑似typewell) で差し替えて A/B。出力は固定ラグ平滑(SMOOTH_LAG)。
import torch, time
V27_DEV = "cuda" if torch.cuda.is_available() else "cpu"; V27_DT = torch.float32
PF_N       = 400
N_SEED     = 32
SMOOTH_LAG = 192     # v25 6.786 と同じ固定ラグ。0=forward PF
MIX_W      = 1.0     # 疑似typewell混合比(covered帯; 1.0=実HWテンプレで完全置換=勝者, 0=typewellのみ)
POOL_MODE  = "median" # 複数参照の疑似typewell生成: "mean" / "median"(平均のボケ回避) / "best1"(最類似1本)
MIX_MAX_REFS = 1      # 混合を適用する参照本数の上限(1=ペアのみ。参照多数群はtypewellが強いので混ぜない=ゲート)
SEED       = 0
CHUNK      = 16      # GPUバッチ坑井数(OOMなら下げる)
print("v27 PF device:", V27_DEV, "| torch:", torch.__version__)

R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}
P0 = {**R0, "n_particles": PF_N, "smooth_lag": SMOOTH_LAG}

def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=V27_DT):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _pad(inps, device, dtype=V27_DT):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s

# --- v27: valid坑井の PF入力 + 尤度grid(typewell / 混合) ---
def build_ctx(va, tw, P=P0):
    md = va["md"]; z = va["z"]; ev = va["eval_mask"]; kn = ~ev
    tw = tw.sort_values("TVT"); tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    tw_gr_pf = _smooth(tw_gr, fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_sm = _smooth(va["gr"], fb, P["hgr_smooth_r"])
    ki = np.where(kn)[0]
    if len(ki) >= 20:
        resid = gr_sm[ki] - np.interp(va["tvt_input"][ki], tw_tvt, tw_gr_pf); gs = float(np.nanstd(resid))
        if not np.isfinite(gs) or gs <= 0: gs = P["gr_sig_def"]
    else:
        gs = P["gr_sig_def"]
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    ls = float(va["tvt_input"][ki[-1]] + z[ki[-1]])
    tl = ki[-30:]; dt = np.diff(va["tvt_input"][tl]); dz = np.diff(z[tl]); dm = np.diff(md[tl]); mm = dm > 0
    ir = float(np.median((dt + dz)[mm] / dm[mm])) if int(mm.sum()) >= 3 else 0.0
    return dict(md=md[ev].astype(np.float64), z=z[ev].astype(np.float64), gr=gr_sm[ev].astype(np.float64),
                gs=gs, ls=ls, ir=ir, gmin=float(gmin), gst=float(gst), gg_tw=np.asarray(gg, np.float64),
                kn_tvt=va["tvt_input"][ki].astype(np.float64), kn_gr=gr_sm[ki].astype(np.float64))  # best1用

def to_inp(ctx, gg, N):
    return dict(md=ctx["md"], z=ctx["z"], gr=ctx["gr"], gg=np.asarray(gg, np.float64),
                gmin=ctx["gmin"], gst=ctx["gst"], gs=ctx["gs"], ls=ctx["ls"], ir=ctx["ir"], N=N)

def _ref_pseudo(r, gmin, step, G):   # 参照1本を疑似typewell(TVTビン平均GR)に
    rt = r["tvt"]; rg = _smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float)
    m = np.isfinite(rt) & np.isfinite(rg); ii = np.round((rt[m] - gmin) / step).astype(int)
    ok = (ii >= 0) & (ii < G); ii, v = ii[ok], rg[m][ok]
    s = np.bincount(ii, weights=v, minlength=G); c = np.bincount(ii, minlength=G)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

def mixed_grid(ctx, refs, w_hw=MIX_W, pool=POOL_MODE):
    gg = ctx["gg_tw"].copy(); gmin = ctx["gmin"]; step = ctx["gst"]; G = len(gg)
    # best1: valid の既知prefix GR を最も再現する参照1本を選ぶ(spatialでなくGR-fit)
    if pool == "best1" and len(refs) > 1:
        kb = np.round((ctx["kn_tvt"] - gmin) / step).astype(int); vk = (kb >= 0) & (kb < G)
        best, bres = refs[0], np.inf
        for r in refs:
            gh = _ref_pseudo(r, gmin, step, G); pred = gh[kb[vk]]
            res = ctx["kn_gr"][vk] - pred; res = res[np.isfinite(res)]
            rr = float(np.sqrt(np.mean(res * res))) if len(res) else np.inf
            if rr < bres: bres, best = rr, r
        refs = [best]
    rt = np.concatenate([r["tvt"] for r in refs])
    rg = np.concatenate([_smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float) for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt, rg = rt[m], rg[m]
    idx = np.round((rt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, rgv = idx[ok], rg[ok]
    if pool == "median":
        ghw = np.full(G, np.nan); med = pd.Series(rgv).groupby(idx).median()
        ghw[med.index.to_numpy()] = med.to_numpy()
    else:  # mean / best1(単一参照)
        sums = np.bincount(idx, weights=rgv, minlength=G); cnts = np.bincount(idx, minlength=G)
        ghw = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan)
    have = np.isfinite(ghw); gm = gg.copy(); gm[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    return gm

def run_batched(inps, P=P0, chunk=CHUNK):
    outs = [None] * len(inps)
    for c0 in range(0, len(inps), chunk):
        _, os_ = run_smoother_ens(inps[c0:c0 + chunk], P, V27_DEV, SEED, N_SEED)
        for j, o in enumerate(os_): outs[c0 + j] = o
        if V27_DEV == "cuda": torch.cuda.empty_cache()
    return outs
print("PF engine (v25 r0 torch/GPU) ready | N=%d n_seed=%d lag=%d mix_w=%.2f" % (PF_N, N_SEED, SMOOTH_LAG, MIX_W))

In [ ]:
# ===== PF A/B (GPUバッチ): typewellのみ vs +疑似typewell混合 =====
PF_MAX_WELLS = None     # 動作確認は 6 等に絞る
_pairs = oof_pairs if PF_MAX_WELLS is None else oof_pairs[:PF_MAX_WELLS]
va_list   = [valid_arrays(p["valid"]) for p in _pairs]
refs_list = [[ref_arrays(r) for r in p["refs"]] for p in _pairs]
true_list = [va["tvt_true"][va["eval_mask"]] for va in va_list]
cov_list  = []
for i in range(len(_pairs)):
    a, b = _ref_pool_tvt(refs_list[i]); cov_list.append((true_list[i] >= a) & (true_list[i] <= b))
ctxs = [build_ctx(va_list[i], TW_CACHE[_pairs[i]["valid"]], P0) for i in range(len(_pairs))]
inps_tw  = [to_inp(ctxs[i], ctxs[i]["gg_tw"], PF_N) for i in range(len(_pairs))]
inps_mix = [to_inp(ctxs[i], mixed_grid(ctxs[i], refs_list[i], MIX_W), PF_N) for i in range(len(_pairs))]

t0 = time.time()
out_tw  = run_batched(inps_tw,  P0)
out_mix = run_batched(inps_mix, P0)
print(f"PF done {time.time()-t0:.0f}s on {V27_DEV} (wells={len(_pairs)}, n_seed={N_SEED}, lag={SMOOTH_LAG})")

preds = {
    "hold":              [np.full(len(true_list[i]), va_list[i]["last_known_tvt"]) for i in range(len(_pairs))],
    "PF typewell":       [np.asarray(out_tw[i],  float)[:len(true_list[i])] for i in range(len(_pairs))],
    "PF +混合(%g)" % MIX_W: [np.asarray(out_mix[i], float)[:len(true_list[i])] for i in range(len(_pairs))],
    "ORACLE上限":        [np.where(cov_list[i], true_list[i], va_list[i]["last_known_tvt"]) for i in range(len(_pairs))],
}
# ゲート: 参照本数<=MIX_MAX_REFS の坑井だけ混合、それ以外は typewell単体(追加PF不要, 既存出力を選択)
_gate = [out_mix[i] if len(refs_list[i]) <= MIX_MAX_REFS else out_tw[i] for i in range(len(_pairs))]
preds["PF 混合gated(<=%dref)" % MIX_MAX_REFS] = [np.asarray(_gate[i], float)[:len(true_list[i])] for i in range(len(_pairs))]
def _pool(name, idxs):
    E = np.concatenate([preds[name][i] - true_list[i] for i in idxs])
    C = np.concatenate([cov_list[i] for i in idxs])
    r = lambda x: float(np.sqrt(np.mean(x*x))) if len(x) else float("nan")
    return r(E), r(E[C]), r(E[~C]), float(C.mean())
subsets = [("全", list(range(len(_pairs)))),
           ("2本ペア", [i for i, p in enumerate(_pairs) if p["group_size"] == 2]),
           ("10本群", [i for i, p in enumerate(_pairs) if p["group_size"] >= 3])]
for sname, idxs in subsets:
    if not idxs: continue
    print(f"[{sname}] valid {len(idxs)}本   {'overall':>8s} {'covered':>8s} {'toe':>8s} {'cov%':>6s}")
    for name in preds:
        o, c, t, cf = _pool(name, idxs)
        print(f"  {name:16s} {o:8.3f} {c:8.3f} {t:8.3f} {cf*100:5.1f}")
    print()
print("※ PF=v25 r0 torch(GPU, 尤度加重seed, 固定ラグ%d)。grid以外はv25と完全同一。" % SMOOTH_LAG)
print("※ 設定: MIX_W=%.2f POOL_MODE=%s。参照多数群は mean=平均ボケ→median/best1 を試す。0=typewellのみ。" % (MIX_W, POOL_MODE))